In [44]:
import numpy as np
import pandas as pd

## 1. Environment Setup & Data Loading


In [45]:
df_customers = pd.read_csv('dataset/olist_customers_dataset.csv')
df_sellers = pd.read_csv('dataset/olist_sellers_dataset.csv')
df_products = pd.read_csv('dataset/olist_products_dataset.csv')
df_orders = pd.read_csv('dataset/olist_orders_dataset.csv')
df_order_items = pd.read_csv('dataset/olist_order_items_dataset.csv')
df_order_payments = pd.read_csv('dataset/olist_order_payments_dataset.csv')
df_product_category_name_translation = pd.read_csv('dataset/product_category_name_translation.csv')

In [46]:
DATASETS = {
    'customers': df_customers,
    'sellers': df_sellers,
    'products': df_products,
    'orders': df_orders,
    'order_items': df_order_items,
    'order_payments': df_order_payments,
    'category_translation': df_product_category_name_translation,
}

PRIMARY_KEYS = {
    'customers': ['customer_id'],
    'sellers': ['seller_id'],
    'products': ['product_id'],
    'orders': ['order_id'],
    'order_items': ['order_id', 'order_item_id'],
    'order_payments': ['order_id', 'payment_sequential'],
    'category_translation': ['product_category_name'],
}

## 2. Dataset Previews


In [47]:
DATASETS['category_translation'].head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [48]:
DATASETS['orders'].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [49]:
DATASETS['order_items'].head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [50]:
DATASETS['sellers'].head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [51]:
DATASETS['order_payments'].head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


## 3. Data Audit Utility Functions


In [52]:
from IPython.display import display


def missing_values(df):
    summary = df.isna().sum().to_frame('missing')
    summary['pct'] = (summary['missing'] / len(df) * 100).round(2)
    return summary.query('missing > 0')


def duplicate_summary(df, keys=None):
    summary = {'full_row_duplicates': df.duplicated().sum()}
    if keys:
        summary['key_duplicates'] = df.duplicated(subset=keys).sum()
        summary['unique_keys'] = df.drop_duplicates(subset=keys).shape[0]
        summary['total_rows'] = len(df)
    return pd.Series(summary)


def numeric_ranges(df):
    numeric = df.select_dtypes(include='number')
    if numeric.empty:
        return None
    return numeric.agg(['min', 'max', 'mean', 'median']).round(2).T


def categorical_overview(df, max_unique=15):
    object_cols = df.select_dtypes(include='object').columns
    if not len(object_cols):
        return None, {}

    overview = pd.DataFrame({
        'nunique': df[object_cols].nunique(dropna=False),
    })
    low_cardinality = {
        col: df[col].value_counts(dropna=False)
        for col in object_cols
        if df[col].nunique(dropna=False) <= max_unique
    }
    return overview, low_cardinality


def whitespace_issues(df):
    rows = []
    for col in df.select_dtypes(include='object').columns:
        series = df[col].dropna().astype(str)
        count = (series != series.str.strip()).sum()
        if count:
            rows.append({'column': col, 'rows_with_whitespace': count})
    return pd.DataFrame(rows)


def run_general_checks(name, df, keys=None):
    print('=' * 60)
    print(name.upper())
    print('=' * 60)
    print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n')

    print('Dtypes:')
    display(df.dtypes.to_frame('dtype'))

    missing = missing_values(df)
    print('Missing values:')
    display(missing if not missing.empty else 'None')

    print('Duplicates & key uniqueness:')
    display(duplicate_summary(df, keys).to_frame('count'))

    numeric = numeric_ranges(df)
    if numeric is not None:
        print('Numeric ranges:')
        display(numeric)

    overview, value_counts = categorical_overview(df)
    if overview is not None:
        print('Categorical columns (unique counts):')
        display(overview)
        for col, counts in value_counts.items():
            print(f'Value counts — {col}:')
            display(counts.to_frame('count'))

    whitespace = whitespace_issues(df)
    print('Whitespace issues:')
    display(whitespace if not whitespace.empty else 'None')
    print()

## 4. Initial Quality Audit Across All Datasets


In [53]:
for name, df in DATASETS.items():
    run_general_checks(name, df, keys=PRIMARY_KEYS.get(name))

CUSTOMERS
Shape: 99,441 rows x 5 columns

Dtypes:


,dtype
customer_id,object
customer_unique_id,object
customer_zip_code_prefix,int64
customer_city,object
customer_state,object


Missing values:


'None'

Duplicates & key uniqueness:


...

,count
full_row_duplicates,0
key_duplicates,0
unique_keys,71
total_rows,71


Categorical columns (unique counts):


,nunique
product_category_name,71
product_category_name_english,71


Whitespace issues:


'None'

## 5. Product Category English Translation


In [54]:
manual_translation = pd.DataFrame({
    'product_category_name': ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'],
    'product_category_name_english': ['pc_gamer', 'kitchen_portables']  # pick your own English label
})

DATASETS['category_translation'] = pd.concat([DATASETS['category_translation'], manual_translation], ignore_index=True)

In [55]:
DATASETS['category_translation'].head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [56]:
DATASETS['products'].head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [57]:
DATASETS['products'] = DATASETS['products'].merge(
    DATASETS['category_translation'],
    on='product_category_name',
    how='left',
)

In [58]:
DATASETS['products'].head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


In [59]:
DATASETS['products']['product_category_name'] = DATASETS['products']['product_category_name_english']
DATASETS['products'].drop(columns=['product_category_name_english'], inplace=True)

In [60]:
DATASETS['products'].head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,art,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,sports_leisure,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,baby,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,housewares,37.0,402.0,4.0,625.0,20.0,17.0,13.0


## 6. Investigating & Handling Uncategorized Products


In [61]:
missing_products = DATASETS['products'][DATASETS['products']['product_category_name'].isna()]

In [62]:
missing_products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0


In [63]:
DATASETS['order_items'][DATASETS['order_items']['product_id'].isin(missing_products['product_id'])]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
123,0046e1d57f4c07c8c92ab26be8c3dfc0,1,ff6caf9340512b8bf6d2a2a6df032cfa,38e6dada03429a47197d5d584d793b41,2017-10-02 15:49:17,7.79,7.78
125,00482f2670787292280e0a8153d82467,1,a9c404971d1a5b1cbc2e4070e02731fd,702835e4b785b67a084280efca355756,2017-02-17 16:18:07,7.60,10.96
132,004f5d8f238e8908e6864b874eda3391,1,5a848e4ab52fd5445cdc07aab1c40e48,c826c40d7b19f62a09e2d7c5e7295ee2,2018-03-06 09:29:25,122.99,15.61
142,0057199db02d1a5ef41bacbf41f8f63b,1,41eee23c25f7a574dfaf8d5c151dbb12,e5a3438891c0bfdb9394643f95273d8e,2018-01-25 09:07:51,20.30,16.79
171,006cb7cafc99b29548d4f412c7f9f493,1,e10758160da97891c2fdcbc35f0f031d,323ce52b5b81df2cd804b017b7f09aa7,2018-02-22 13:35:28,56.00,14.14
...,...,...,...,...,...,...,...
112306,ff24fec69b7f3d30f9dc1ab3aee7c179,1,5a848e4ab52fd5445cdc07aab1c40e48,c826c40d7b19f62a09e2d7c5e7295ee2,2018-02-01 02:40:12,122.99,15.61
112333,ff3024474be86400847879103757d1fd,1,f9b1795281ce51b1cf39ef6d101ae8ab,3771c85bac139d2344864ede5d9341e3,2017-11-21 03:55:39,39.90,9.94
112350,ff3a45ee744a7c1f8096d2e72c1a23e4,1,b61d1388a17e3f547d2bc218df02335b,07017df32dc5f2f1d2801e579548d620,2017-05-10 10:15:19,139.00,21.42
112438,ff7b636282b98e0aa524264b295ed928,1,431df35e52c10451171d8037482eeb43,6cd68b3ed6d59aaa9fece558ad360c0a,2018-02-22 15:35:35,49.90,15.11


In [64]:
products = DATASETS['products']
order_items = DATASETS['order_items']

missing_category_products = products[
    products['product_category_name'].isna()
]

missing_product_ids = missing_category_products['product_id']

affected_items = order_items[
    order_items['product_id'].isin(missing_product_ids)
]

print("Missing-category products:", len(missing_category_products))
print("Affected order items:", len(affected_items))
print("Affected revenue:", affected_items['price'].sum())

Missing-category products: 610
Affected order items: 1603
Affected revenue: 179535.28


In [65]:
affected_items.shape[0] / len(order_items) * 100

1.4229915667998225

In [66]:
DATASETS['products'] = (
    DATASETS['products']
    .fillna('unknown')
)

## 7. Product Weight & Dimensions Quality Checks


In [67]:
numeric_cols = [
    'product_weight_g', 'product_length_cm',
    'product_height_cm', 'product_width_cm'
]

zero_weight = DATASETS['products'][DATASETS['products']['product_weight_g'] == 0]
zero_weight.head(10)


zero_weight[numeric_cols + ['product_category_name']]

,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name
9769,0.0,30.0,25.0,30.0,bed_bath_table
13683,0.0,30.0,25.0,30.0,bed_bath_table
14997,0.0,30.0,25.0,30.0,bed_bath_table
32079,0.0,30.0,25.0,30.0,bed_bath_table


In [68]:
zero_weight_ids = zero_weight['product_id']

DATASETS['order_items'][
    DATASETS['order_items']['product_id'].isin(zero_weight_ids)
]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
2972,06afc1144eb9f51ef2aa90ec9223c7f4,1,e673e90efa65a5409ff4196c038bb5af,b39d7fe263ef469605dbb32608aee0af,2018-08-23 17:25:20,129.9,23.71
2973,06afc1144eb9f51ef2aa90ec9223c7f4,2,e673e90efa65a5409ff4196c038bb5af,b39d7fe263ef469605dbb32608aee0af,2018-08-23 17:25:20,129.9,23.71
3052,06d9e69034388abf6da64378e10737b8,1,36ba42dd187055e1fbe943b2d11430ca,b39d7fe263ef469605dbb32608aee0af,2018-08-10 09:10:11,100.0,23.85
3053,06d9e69034388abf6da64378e10737b8,2,36ba42dd187055e1fbe943b2d11430ca,b39d7fe263ef469605dbb32608aee0af,2018-08-10 09:10:11,100.0,23.85
14080,200b121c28e10ef638131a7c76753327,1,81781c0fed9fe1ad6e8c81fca1e1cb08,b39d7fe263ef469605dbb32608aee0af,2018-08-14 16:10:16,100.0,19.89
31488,476b812a7e4fc972646eb390517bddcb,1,e673e90efa65a5409ff4196c038bb5af,b39d7fe263ef469605dbb32608aee0af,2018-08-22 11:30:42,129.9,23.71
32984,4abc7b5330425bcf9c2f7f48151a88c0,1,8038040ee2a71048d4bdbbdc985b69ab,b39d7fe263ef469605dbb32608aee0af,2018-08-09 21:31:33,129.9,14.49
79374,b489f7ae130ba3fd26b0a20f8cc81c61,1,e673e90efa65a5409ff4196c038bb5af,b39d7fe263ef469605dbb32608aee0af,2018-08-22 20:49:57,129.9,23.71


In [69]:
DATASETS['products'].assign(zero_weight=DATASETS['products']['product_weight_g'].eq(0)).groupby('zero_weight')['product_id'].count()

zero_weight
False    32947
True         4
Name: product_id, dtype: int64

In [70]:
DATASETS['products'][DATASETS['products']['product_weight_g'] == 0]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,bed_bath_table,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,bed_bath_table,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,bed_bath_table,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,bed_bath_table,53.0,528.0,1.0,0.0,30.0,25.0,30.0


## 8. Orders Dataset Inspection & Missing Values Analysis


In [71]:
for name, df in DATASETS.items():
    run_general_checks(name, df, keys=PRIMARY_KEYS.get(name))

CUSTOMERS
Shape: 99,441 rows x 5 columns

Dtypes:


,dtype
customer_id,object
customer_unique_id,object
customer_zip_code_prefix,int64
customer_city,object
customer_state,object


Missing values:


'None'

Duplicates & key uniqueness:


...

,count
full_row_duplicates,0
key_duplicates,0
unique_keys,73
total_rows,73


Categorical columns (unique counts):


,nunique
product_category_name,73
product_category_name_english,73


Whitespace issues:


'None'

In [72]:
DATASETS['orders'][DATASETS['orders']['order_delivered_customer_date'].isna()]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00
44,ee64d42b8cf066f35eac1cf57de1aa85,caded193e8e47b8362864762a83db3c5,shipped,2018-06-04 16:44:48,2018-06-05 04:31:18,2018-06-05 14:32:00,NaN,2018-06-28 00:00:00
103,0760a852e4e9d89eb77bf631eaaf1c84,d2a79636084590b7465af8ab374a8cf5,invoiced,2018-08-03 17:44:42,2018-08-07 06:15:14,NaN,NaN,2018-08-21 00:00:00
128,15bed8e2fec7fdbadb186b57c46c92f2,f3f0e613e0bdb9c7cee75504f0f90679,processing,2017-09-03 14:22:03,2017-09-03 14:30:09,NaN,NaN,2017-10-03 00:00:00
154,6942b8da583c2f9957e990d028607019,52006a9383bf149a4fb24226b173106f,shipped,2018-01-10 11:33:07,2018-01-11 02:32:30,2018-01-11 19:39:23,NaN,2018-02-07 00:00:00
...,...,...,...,...,...,...,...,...
99283,3a3cddda5a7c27851bd96c3313412840,0b0d6095c5555fe083844281f6b093bb,canceled,2018-08-31 16:13:44,NaN,NaN,NaN,2018-10-01 00:00:00
99313,e9e64a17afa9653aacf2616d94c005b8,b4cd0522e632e481f8eaf766a2646e86,processing,2018-01-05 23:07:24,2018-01-09 07:18:05,NaN,NaN,2018-02-06 00:00:00
99347,a89abace0dcc01eeb267a9660b5ac126,2f0524a7b1b3845a1a57fcf3910c4333,canceled,2018-09-06 18:45:47,NaN,NaN,NaN,2018-09-27 00:00:00
99348,a69ba794cc7deb415c3e15a0a3877e69,726f0894b5becdf952ea537d5266e543,unavailable,2017-08-23 16:28:04,2017-08-28 15:44:47,NaN,NaN,2017-09-15 00:00:00


In [73]:
DATASETS['orders'][DATASETS['orders'].isna().any(axis=1)]['order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered        23
created           5
approved          2
Name: count, dtype: int64

In [74]:
null_cols = ['order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date']

null_vs_status = (
    df_orders
    .assign(**{col: df_orders[col].isna() for col in null_cols})
    .groupby('order_status')[null_cols]
    .sum()
    .astype(int)
)
null_vs_status['total_orders'] = df_orders.groupby('order_status').size()
null_vs_status

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,total_orders
order_status,,,,
approved,0,2,2,2
canceled,141,550,619,625
created,5,5,5,5
delivered,14,2,8,96478
invoiced,0,314,314,314
processing,0,301,301,301
shipped,0,0,1107,1107
unavailable,0,609,609,609


### Handling Incomplete Delivered Orders


In [75]:
mask = (
    (df_orders['order_status'] == 'delivered') &
    (df_orders[['order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date']].isna().any(axis=1))
)
df_orders = df_orders[~mask]
DATASETS['orders'] = df_orders

### Datetime Conversions & Chronological Sequence Validation


In [76]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    df_orders[col] = pd.to_datetime(df_orders[col])
df_order_items['shipping_limit_date'] = pd.to_datetime(df_order_items['shipping_limit_date'])

invalid_approved = df_orders[df_orders['order_approved_at'] < df_orders['order_purchase_timestamp']]
invalid_carrier = df_orders[df_orders['order_delivered_carrier_date'] < df_orders['order_approved_at']]
invalid_customer = df_orders[df_orders['order_delivered_customer_date'] < df_orders['order_delivered_carrier_date']]

print(f"invalid_approved: {len(invalid_approved)}")
print(f"invalid_carrier: {len(invalid_carrier)}")
print(f"invalid_customer: {len(invalid_customer)}")


invalid_approved: 0
invalid_carrier: 1359
invalid_customer: 23


### Financial & Order Payment Consistency Checks


In [77]:
zero_or_neg_items = df_order_items[(df_order_items['price'] <= 0) | (df_order_items['freight_value'] < 0)]
zero_or_neg_payments = df_order_payments[df_order_payments['payment_value'] <= 0]

items_total = df_order_items.groupby('order_id')[['price', 'freight_value']].sum().sum(axis=1).rename('items_total')
payments_total = df_order_payments.groupby('order_id')['payment_value'].sum().rename('payments_total')
financial_comparison = pd.concat([items_total, payments_total], axis=1).dropna()
financial_comparison['diff'] = (financial_comparison['items_total'] - financial_comparison['payments_total']).round(2)
mismatched_orders = financial_comparison[financial_comparison['diff'] != 0]

print(f"zero_or_neg_items: {len(zero_or_neg_items)}")
print(f"zero_or_neg_payments: {len(zero_or_neg_payments)}")
print(f"mismatched_orders: {len(mismatched_orders)}")


zero_or_neg_items: 0
zero_or_neg_payments: 9
mismatched_orders: 576


### Referential Integrity Validation


In [78]:
missing_order_items_orders = set(df_order_items['order_id']) - set(df_orders['order_id'])
missing_order_payments_orders = set(df_order_payments['order_id']) - set(df_orders['order_id'])
missing_products = set(df_order_items['product_id']) - set(df_products['product_id'])
missing_sellers = set(df_order_items['seller_id']) - set(df_sellers['seller_id'])
missing_customers = set(df_orders['customer_id']) - set(df_customers['customer_id'])

print(f"missing_order_items_orders: {len(missing_order_items_orders)}")
print(f"missing_order_payments_orders: {len(missing_order_payments_orders)}")
print(f"missing_products: {len(missing_products)}")
print(f"missing_sellers: {len(missing_sellers)}")
print(f"missing_customers: {len(missing_customers)}")


missing_order_items_orders: 23
missing_order_payments_orders: 23
missing_products: 0
missing_sellers: 0
missing_customers: 0


### Geographic & Text Standardization


In [79]:
df_customers['customer_city'] = df_customers['customer_city'].str.strip().str.lower()
df_sellers['seller_city'] = df_sellers['seller_city'].str.strip().str.lower()

df_customers['customer_state'] = df_customers['customer_state'].str.strip().str.upper()
df_sellers['seller_state'] = df_sellers['seller_state'].str.strip().str.upper()

print(f"customer_cities_unique: {df_customers['customer_city'].nunique()}")
print(f"seller_cities_unique: {df_sellers['seller_city'].nunique()}")
print(f"customer_states_unique: {df_customers['customer_state'].nunique()}")
print(f"seller_states_unique: {df_sellers['seller_state'].nunique()}")


customer_cities_unique: 4119
seller_cities_unique: 611
customer_states_unique: 27
seller_states_unique: 23
